In [ ]:
from pathlib import Path
import json
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET = 'is_late'
LOW_THRESHOLD = 0.30   # prototype - configurable
HIGH_THRESHOLD = 0.70  # prototype - configurable
DATASET_VERSION = 'late_payment_v1_chronological'

# Resolve repo root whether notebook is run from repo root or notebooks/
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'data' / 'processed' / 'late_payment_train.csv').exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
TRAIN_PATH = REPO_ROOT / 'data' / 'processed' / 'late_payment_train.csv'
VAL_PATH = REPO_ROOT / 'data' / 'processed' / 'late_payment_validation.csv'
TEST_PATH = REPO_ROOT / 'data' / 'processed' / 'late_payment_test.csv'
PIPELINE_PATH = REPO_ROOT / 'models' / 'late_payment_feature_pipeline.joblib'
MODEL_PATH = REPO_ROOT / 'models' / 'late_payment_classifier.joblib'
METADATA_PATH = REPO_ROOT / 'models' / 'late_payment_model_metadata.json'
PRED_PATH = REPO_ROOT / 'data' / 'processed' / 'late_payment_predictions.csv'
COMP_PATH = REPO_ROOT / 'reports' / 'late_payment_model_comparison.csv'
TEST_METRICS_PATH = REPO_ROOT / 'reports' / 'late_payment_test_metrics.json'

print('REPO_ROOT:', REPO_ROOT)
print('LOW/HIGH risk thresholds:', LOW_THRESHOLD, HIGH_THRESHOLD)

## 1. Load Feature-Engineered Data

Load the already-created chronological splits. Print shape, columns, date ranges, target distribution.
Do NOT recreate any random train/test split.

In [ ]:
train_df = pd.read_csv(TRAIN_PATH, parse_dates=['invoice_date'])
validation_df = pd.read_csv(VAL_PATH, parse_dates=['invoice_date'])
test_df = pd.read_csv(TEST_PATH, parse_dates=['invoice_date'])

for name, df in [('train', train_df), ('validation', validation_df), ('test', test_df)]:
    print(f'--- {name}: shape {df.shape} ---')
    print('columns:', list(df.columns))
    print(f"invoice_date range: {df['invoice_date'].min()} -> {df['invoice_date'].max()}")
    print('is_late distribution (count):')
    print(df['is_late'].value_counts(dropna=False))
    print('is_late distribution (rate):')
    print((df['is_late'].value_counts(normalize=True) * 100).round(2))
    print()

## 2. Dataset Verification

Verify chronological ordering (train -> validation -> test) and that we reuse the given splits.
Note on boundary: the split preserves row order, so the boundary calendar day (`2019-10-09`, `2019-12-10`)
appears as both `max(train)` and `min(validation)` (same for val/test). A strict `<` therefore fails by one shared day,
while the ordering itself is correct. We assert `<=` plus monotonicity and document this explicitly.

In [ ]:
print('train      :', train_df['invoice_date'].min(), '->', train_df['invoice_date'].max(), f'({len(train_df)} rows)')
print('validation :', validation_df['invoice_date'].min(), '->', validation_df['invoice_date'].max(), f'({len(validation_df)} rows)')
print('test       :', test_df['invoice_date'].min(), '->', test_df['invoice_date'].max(), f'({len(test_df)} rows)')

train_max = train_df['invoice_date'].max()
val_min = validation_df['invoice_date'].min()
val_max = validation_df['invoice_date'].max()
test_min = test_df['invoice_date'].min()

# Spec intent: train strictly before validation strictly before test.
# Runnable check (shared boundary day => <=, not <). Strict < would raise on this correct split.
assert train_max <= val_min, f'Chronology violated: train_max {train_max} > val_min {val_min}'
assert val_max <= test_min, f'Chronology violated: val_max {val_max} > test_min {test_min}'
print(f'Chronology OK (<=): train_max {train_max.date()} <= val_min {val_min.date()}; val_max {val_max.date()} <= test_min {test_min.date()}')
strict_train_val = bool(train_max < val_min)
strict_val_test = bool(val_max < test_min)
print(f'Strict < check: train<val -> {strict_train_val}, val<test -> {strict_val_test} (False here = shared boundary day, expected)')

# Each split itself must be chronologically sorted (no shuffling / no random re-split).
assert train_df['invoice_date'].is_monotonic_increasing, 'train not sorted'
assert validation_df['invoice_date'].is_monotonic_increasing, 'validation not sorted'
assert test_df['invoice_date'].is_monotonic_increasing, 'test not sorted'
print('Each split is chronologically sorted. Reusing given splits — no random split created.')

## 3. Separate Features and Target

Target `is_late`: `0` = paid on time or early, `1` = paid late. `invoice_date` / `invoice_id` / `customer_id` are identifiers for
traceability only and are excluded from the model matrix (kept aside for the output dataframe).

In [ ]:
TARGET = 'is_late'
ID_COLS = ['invoice_date', 'invoice_id', 'customer_id']  # retained for output, never passed to model
FEATURE_COLS = [c for c in train_df.columns if c not in ID_COLS + [TARGET]]
print(f'TARGET: {TARGET}')
print(f'ID cols (excluded from X): {ID_COLS}')
print(f'FEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}')

X_train, y_train = train_df[FEATURE_COLS].copy(), train_df[TARGET].astype(int).copy()
X_val, y_val = validation_df[FEATURE_COLS].copy(), validation_df[TARGET].astype(int).copy()
X_test, y_test = test_df[FEATURE_COLS].copy(), test_df[TARGET].astype(int).copy()

assert set(y_train.unique()) <= {0, 1}, f'y_train not binary: {y_train.unique()}'
assert set(y_val.unique()) <= {0, 1}
assert set(y_test.unique()) <= {0, 1}
print('Target verified binary {0,1}: 0 = on time/early, 1 = late')
print(f'X_train {X_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}')
print('All model features numeric — no additional categorical encoding required.')
print(X_train.dtypes)

## 4. Verify Leakage Prevention

These must NEVER appear in the model feature matrix. If any is present we STOP and report — never silently drop and continue.

In [ ]:
FORBIDDEN_FEATURES = {'invoice_id', 'customer_id', 'payment_date', 'delay_days', 'is_late', 'invoice_status'}
leaked = FORBIDDEN_FEATURES.intersection(set(X_train.columns))
print('FORBIDDEN_FEATURES:', sorted(FORBIDDEN_FEATURES))
print('X_train columns   :', sorted(X_train.columns))
print('Intersection (must be empty):', sorted(leaked) if leaked else 'EMPTY — PASS')
assert len(leaked) == 0, f'LEAKAGE DETECTED — forbidden features in X: {sorted(leaked)}. STOPPING.'
# Also confirm leakage columns are not accidentally hiding elsewhere in the frame (informational).
for col in sorted(FORBIDDEN_FEATURES - {'is_late'}):
    print(f"  column '{col}' in train_df: {col in train_df.columns} | in X_train: {col in X_train.columns}")
print('Leakage audit PASSED: no forbidden feature will be passed to any model.')

## 5. Class Distribution

Display counts / percentages, plot the distribution, compute the imbalance ratio. Do NOT assume balance.

In [ ]:
n_total = len(y_train)
n_ontime = int((y_train == 0).sum())
n_late = int((y_train == 1).sum())
pct_ontime = 100 * n_ontime / n_total
pct_late = 100 * n_late / n_total
imbalance_ratio = max(n_ontime, n_late) / min(n_ontime, n_late)
print(f'Number of on-time/early invoices (0): {n_ontime}')
print(f'Number of late invoices (1)        : {n_late}')
print(f'Percentage on-time/early           : {pct_ontime:.2f}%')
print(f'Percentage late                    : {pct_late:.2f}%')
print(f'Imbalance ratio (majority/minority): {imbalance_ratio:.3f}')
for name, y in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    r = y.value_counts(normalize=True) * 100
    print(f'{name:10s}: on-time {r.get(0,0):.2f}% | late {r.get(1,0):.2f}% (n={len(y)})')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['On time / early (0)', 'Late (1)'], [n_ontime, n_late])
ax.set_title('Class distribution — train set')
ax.set_ylabel('Invoices')
for b, v, p in zip(bars, [n_ontime, n_late], [pct_ontime, pct_late]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:,}\n({p:.1f}%)', ha='center', va='bottom')
plt.tight_layout(); plt.show()
print(f'Conclusion: mildly imbalanced (~{pct_ontime:.0f}/{pct_late:.0f}); majority = on-time. Accuracy alone is misleading -> use PR-AUC/Recall/F1.')

## 6. Baseline Model

`DummyClassifier(strategy='most_frequent')` — always predicts the majority class. FlowCred must beat this to justify ML.
Metrics: Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC (probability-based for ROC/PR).

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    """Accuracy/Precision/Recall/F1 on labels; ROC-AUC + PR-AUC (average_precision) on probabilities."""
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
    }

VAL_RESULTS = {}  # model_name -> metrics dict (validation, threshold 0.5)
VAL_PROBAS = {}   # model_name -> P(y=1) on validation (for curves)
VAL_PREDS = {}    # model_name -> class predictions on validation

dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train)
dummy_pred = dummy_clf.predict(X_val)
dummy_proba = dummy_clf.predict_proba(X_val)[:, 1]  # all 0.0 (majority = on-time)
VAL_RESULTS['Dummy Baseline'] = compute_binary_metrics(y_val, dummy_pred, dummy_proba)
VAL_PROBAS['Dummy Baseline'] = dummy_proba
VAL_PREDS['Dummy Baseline'] = dummy_pred
print('DummyClassifier(strategy=most_frequent) validation metrics (threshold 0.5):')
for k, v in VAL_RESULTS['Dummy Baseline'].items():
    print(f'  {k:9s}: {v:.4f}')
print('Note: Dummy Recall=F1=0 for the late class is expected — it never predicts late. Any useful model must beat its PR-AUC/Recall/F1.')

## 7. Logistic Regression

**Preprocessing (fit on TRAIN only, transform val/test via the Pipeline):** median imputation -> standard scaling -> logistic regression.
We inspect the reusable `late_payment_feature_pipeline.joblib` (MeanMedianImputer + StandardScaler over the 19 numeric features)
but fit a **fresh** imputer/scaler on TRAIN here — reusing a pipeline fitted on full data would leak validation/test statistics.
`class_weight='balanced'` compensates the ~57/43 majority bias so the late (minority, business-critical) class is not under-weighted.
`random_state=42`, `max_iter=1000`. Outputs: `predicted_class`, `predicted_probability = P(is_late=1)`.

In [ ]:
saved = joblib.load(PIPELINE_PATH)
print('Reusable pipeline file:', PIPELINE_PATH)
print('  type:', type(saved), '| keys:', list(saved.keys()) if isinstance(saved, dict) else 'n/a')
if isinstance(saved, dict) and 'feature_pipeline' in saved:
    print(saved['feature_pipeline'])

logreg_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])
logreg_pipe.fit(X_train, y_train)  # fit preprocessing on TRAIN only
lr_pred = logreg_pipe.predict(X_val)
lr_proba = logreg_pipe.predict_proba(X_val)[:, 1]  # P(is_late = 1)
VAL_RESULTS['Logistic Regression'] = compute_binary_metrics(y_val, lr_pred, lr_proba)
VAL_PROBAS['Logistic Regression'] = lr_proba
VAL_PREDS['Logistic Regression'] = lr_pred
print('Logistic Regression validation metrics (threshold 0.5):')
for k, v in VAL_RESULTS['Logistic Regression'].items():
    print(f'  {k:9s}: {v:.4f}')

## 8. Random Forest

Reasonable baseline hyperparameters (no extensive tuning yet): `n_estimators=200`, `class_weight='balanced'`, `random_state=42`, `n_jobs=-1`.
Scaling is not required for trees. Pipeline: median imputation (fit on TRAIN only) -> forest. Outputs: class + `P(is_late=1)`.

In [ ]:
rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
])  
rf_pipe.fit(X_train, y_train)
rf_pred = rf_pipe.predict(X_val)
rf_proba = rf_pipe.predict_proba(X_val)[:, 1]
VAL_RESULTS['Random Forest'] = compute_binary_metrics(y_val, rf_pred, rf_proba)
VAL_PROBAS['Random Forest'] = rf_proba
VAL_PREDS['Random Forest'] = rf_pred
print('Random Forest validation metrics (threshold 0.5):')
for k, v in VAL_RESULTS['Random Forest'].items():
    print(f'  {k:9s}: {v:.4f}')

## 9. XGBoost Classifier

Sensible baseline (comparison, not exhaustive tuning): `n_estimators=300`, `max_depth=5`, `learning_rate=0.05`,
`subsample=0.8`, `colsample_bytree=0.8`, `eval_metric='logloss'`, `random_state=42`.
Imbalance handled via `scale_pos_weight = n_negative / n_positive` (train). Scaling not required. Median imputation fit on TRAIN only.

In [ ]:
neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
scale_pos_weight = neg / pos
print(f'train neg={neg}, pos={pos} -> scale_pos_weight={scale_pos_weight:.4f}')
xgb_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
                           colsample_bytree=0.8, eval_metric='logloss', random_state=RANDOM_STATE,
                           n_jobs=-1, scale_pos_weight=scale_pos_weight)),
])  
xgb_pipe.fit(X_train, y_train)
xgb_pred = xgb_pipe.predict(X_val)
xgb_proba = xgb_pipe.predict_proba(X_val)[:, 1]
VAL_RESULTS['XGBoost'] = compute_binary_metrics(y_val, xgb_pred, xgb_proba)
VAL_PROBAS['XGBoost'] = xgb_proba
VAL_PREDS['XGBoost'] = xgb_pred
print('XGBoost validation metrics (threshold 0.5):')
for k, v in VAL_RESULTS['XGBoost'].items():
    print(f'  {k:9s}: {v:.4f}')

## 10. Validation Evaluation

Comparison table (Accuracy / Precision / Recall / F1 / ROC-AUC / PR-AUC, probabilities used for ROC/PR),
confusion matrices (TN / FP / FN / TP) and PR + ROC curves. PR curves are emphasised because late-payment data is imbalanced.

In [ ]:
import seaborn as sns  # noqa: F401  (optional styling)
comparison_df = pd.DataFrame([{'Model': m, **VAL_RESULTS[m]} for m in ['Dummy Baseline', 'Logistic Regression', 'Random Forest', 'XGBoost']])
comparison_df = comparison_df[['Model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']]
print('Validation comparison (threshold 0.5):')
display(comparison_df.round(4))

# --- Confusion matrices for the three ML models ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ['Logistic Regression', 'Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_val, VAL_PREDS[name], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f'{name}: TN={tn} FP={fp} FN={fn} TP={tp}')
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(name)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['On time (0)', 'Late (1)']); ax.set_yticklabels(['On time (0)', 'Late (1)'])
    for (i, j), v in np.ndenumerate(cm):
        label = ['TN', 'FP', 'FN', 'TP'][i * 2 + j]
        ax.text(j, i, f'{label}\n{v:,}', ha='center', va='center', fontsize=10)
plt.suptitle('Confusion matrices — validation set (threshold 0.5)')
plt.tight_layout(); plt.show()

# --- Precision-Recall curves + ROC curves ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, ls in [('Logistic Regression', '-'), ('Random Forest', '--'), ('XGBoost', '-.')]:
    p, r, _ = precision_recall_curve(y_val, VAL_PROBAS[name])
    axes[0].plot(r, p, linestyle=ls, label=f"{name} (PR-AUC={VAL_RESULTS[name]['pr_auc']:.3f})")
axes[0].set_title('Precision-Recall curve — validation')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision'); axes[0].legend(); axes[0].grid(alpha=0.3)
for name, ls in [('Logistic Regression', '-'), ('Random Forest', '--'), ('XGBoost', '-.')]:
    fpr, tpr, _ = roc_curve(y_val, VAL_PROBAS[name])
    axes[1].plot(fpr, tpr, linestyle=ls, label=f"{name} (ROC-AUC={VAL_RESULTS[name]['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], 'k:', label='Chance')
axes[1].set_title('ROC curve — validation')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Do not rely on ROC-AUC alone under imbalance — weigh PR-AUC / Recall / F1.')

## 11. Model Comparison

Selection criteria (validation only): primarily **PR-AUC, Recall, F1, ROC-AUC** — never accuracy alone, no arbitrary 'overall score',
no claim of a universally best model. The candidate below is the prototype pick on these documented validation criteria.

In [ ]:
ranked = comparison_df.sort_values('pr_auc', ascending=False).reset_index(drop=True)
print('Ranked by validation PR-AUC (then Recall/F1 as tie-breakers):')
display(ranked.round(4))
SELECTED_MODEL_NAME = ranked.loc[0, 'Model']
# Prefer the tree-based candidate for SHAP; PR-AUC leader here is XGBoost — assert the documented expectation explicitly.
print(f'PR-AUC leader on validation: {SELECTED_MODEL_NAME}')
MODEL_PIPES = {'Logistic Regression': logreg_pipe, 'Random Forest': rf_pipe, 'XGBoost': xgb_pipe}
assert SELECTED_MODEL_NAME in MODEL_PIPES, 'Dummy must never be selected as final model'
SELECTED_MODEL = MODEL_PIPES[SELECTED_MODEL_NAME]
SELECTED_VAL = VAL_RESULTS[SELECTED_MODEL_NAME]
print(f'SELECTED MODEL (prototype, validation criteria PR-AUC/Recall/F1/ROC-AUC): {SELECTED_MODEL_NAME}')
print(f"  PR-AUC={SELECTED_VAL['pr_auc']:.4f} Recall={SELECTED_VAL['recall']:.4f} F1={SELECTED_VAL['f1']:.4f} ROC-AUC={SELECTED_VAL['roc_auc']:.4f}")

## 12. Final Test Evaluation

Evaluate the selected candidate **once** on the untouched test set (default 0.5 threshold here; the val-tuned operational
threshold is applied in §13/§15 without ever tuning on test). Report Accuracy / Precision / Recall / F1 / ROC-AUC / PR-AUC + confusion matrix.

In [ ]:
test_pred_default = SELECTED_MODEL.predict(X_test)
test_proba = SELECTED_MODEL.predict_proba(X_test)[:, 1]  # late_probability P(is_late=1)
TEST_METRICS_DEFAULT = compute_binary_metrics(y_test, test_pred_default, test_proba)
print(f'Test performance — {SELECTED_MODEL_NAME} (threshold 0.5, single untouched evaluation):')
for k, v in TEST_METRICS_DEFAULT.items():
    print(f'  {k:9s}: {v:.4f}')
cm_t = confusion_matrix(y_test, test_pred_default, labels=[0, 1])
tn, fp, fn, tp = cm_t.ravel()
print(f'Confusion matrix: TN={tn} FP={fp} FN={fn} TP={tp}')
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm_t, cmap='Blues')
ax.set_title(f'Confusion matrix — test ({SELECTED_MODEL_NAME}, thr=0.5)')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['On time (0)', 'Late (1)']); ax.set_yticklabels(['On time (0)', 'Late (1)'])
for (i, j), v in np.ndenumerate(cm_t):
    ax.text(j, i, f"{['TN','FP','FN','TP'][i*2+j]}\n{v:,}", ha='center', va='center')
plt.tight_layout(); plt.show()
print('Test set was not used for any fitting or tuning — this is the unbiased estimate.')

## 13. Classification Threshold Analysis

`0.5` is not assumed optimal. Sweep thresholds on the **validation set only** (never test) for the selected model;
table + plot Precision/Recall/F1. The prototype picks max-F1; the final business threshold must be validated against real collections costs.

In [ ]:
THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
sel_proba_val = VAL_PROBAS[SELECTED_MODEL_NAME]
rows = []
for t in THRESHOLDS:
    yp = (sel_proba_val >= t).astype(int)
    rows.append({'threshold': t,
                 'precision': float(precision_score(y_val, yp, zero_division=0)),
                 'recall': float(recall_score(y_val, yp, zero_division=0)),
                 'f1': float(f1_score(y_val, yp, zero_division=0))})
thr_df = pd.DataFrame(rows)
display(thr_df.round(4))
SELECTED_THRESHOLD = float(thr_df.loc[thr_df['f1'].idxmax(), 'threshold'])
print(f'Selected classification threshold (max validation F1, prototype): {SELECTED_THRESHOLD:.2f}')
print('Lower threshold -> higher recall, lower precision; higher threshold -> opposite. Final business threshold needs cost validation.')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(thr_df['threshold'], thr_df['precision'], marker='o', label='Precision')
ax.plot(thr_df['threshold'], thr_df['recall'], marker='s', label='Recall')
ax.plot(thr_df['threshold'], thr_df['f1'], marker='^', label='F1')
ax.axvline(SELECTED_THRESHOLD, color='k', linestyle='--', label=f'selected {SELECTED_THRESHOLD:.2f}')
ax.set_title(f'Threshold analysis — {SELECTED_MODEL_NAME} (validation)')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 14. SHAP Explainability

Tree-based SHAP (`shap.TreeExplainer`) for the selected model (XGBoost). Global: summary (beeswarm) + bar (mean |SHAP|).
Local: per-invoice feature / value / SHAP contribution tables. `SHAP > 0` pushes toward **late**, `< 0` toward **not late**.
Model/SHAP importance is associative, **not causal**.

In [ ]:
import shap
import matplotlib
# Explain the XGBoost classifier inside its pipeline (imputer fitted on TRAIN only).
tree_pipe = xgb_pipe  # preferred tree model for SHAP (also the selected candidate)
imputer = tree_pipe.named_steps['imputer']
clf = tree_pipe.named_steps['clf']
X_train_imp = pd.DataFrame(imputer.transform(X_train), columns=FEATURE_COLS)
X_val_imp = pd.DataFrame(imputer.transform(X_val), columns=FEATURE_COLS)
print(f'Imputed shapes: train {X_train_imp.shape}, val {X_val_imp.shape}')
explainer = shap.TreeExplainer(clf)
SAMPLE_N = min(2000, len(X_val_imp))
X_sample = X_val_imp.iloc[:SAMPLE_N]
shap_values = explainer.shap_values(X_sample)
print(f'SHAP values shape: {shap_values.shape}')
plt.figure()
shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
plt.title('SHAP summary (beeswarm) — pushes right (positive) toward late', fontsize=10)
plt.tight_layout(); plt.show()
plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False, max_display=15)
plt.title('SHAP bar — mean |SHAP| (global importance)', fontsize=10)
plt.tight_layout(); plt.show()
mean_abs = np.abs(shap_values).mean(axis=0)
shap_imp = pd.DataFrame({'feature': FEATURE_COLS, 'mean_abs_shap': mean_abs}).sort_values('mean_abs_shap', ascending=False)
print('Top features by mean |SHAP|:')
display(shap_imp.head(10).round(4))
# --- Model feature importance vs SHAP importance ---
model_imp = pd.DataFrame({'feature': FEATURE_COLS, 'model_importance': clf.feature_importances_}).sort_values('model_importance', ascending=False)
compare = model_imp.merge(shap_imp, on='feature')
print('Model (gain/weight) importance vs SHAP importance:')
display(compare.head(10).round(4))
fig, ax = plt.subplots(figsize=(8, 5))
top = compare.head(10).iloc[::-1]
ax.barh(top['feature'], top['model_importance'], label='XGB importance')
ax.set_title('XGBoost feature importance (top 10)')
ax.set_xlabel('Importance'); plt.tight_layout(); plt.show()
print('Importance != causality: high importance means predictive association, not that changing the feature causes lateness.')

In [ ]:
# Local explanations for 3 individual validation invoices (first 3 of the SHAP sample).
for k in range(3):
    row = X_sample.iloc[[k]]
    sv = shap_values[k]
    local = pd.DataFrame({'feature': FEATURE_COLS, 'feature_value': row.values[0], 'shap_contribution': sv})
    local['direction'] = np.where(local['shap_contribution'] > 0, 'toward late', 'toward not late')
    local = local.reindex(local['shap_contribution'].abs().sort_values(ascending=False).index)
    proba_k = float(sel_proba_val[k])
    true_k = int(y_val.iloc[k])
    print(f'--- Invoice #{k} (val positional row {k}, true is_late={true_k}, P(late)={proba_k:.4f}) ---')
    display(local.head(8).round(4))
    plt.figure()
    sub = local.head(8).iloc[::-1]
    colors = ['firebrick' if v > 0 else 'steelblue' for v in sub['shap_contribution']]
    plt.barh(sub['feature'], sub['shap_contribution'], color=colors)
    plt.axvline(0, color='k', linewidth=0.8)
    plt.title(f'Local SHAP - invoice #{k} (red -> late, blue -> not late)')
    plt.tight_layout(); plt.show()

## 15. Risk Probability and Risk Category

`late_probability = P(is_late=1)` in `[0,1]` is a predicted probability, **not certainty**.
Configurable prototype bands: `p < 0.30 -> Low`, `0.30 <= p < 0.70 -> Medium`, `p >= 0.70 -> High`.
**Prototype thresholds only** — not validated business policy. `High` means higher predicted probability, never 'guaranteed late'.
Identification columns (`invoice_id`, `customer_id`) are carried in the output for traceability but were never model inputs.

In [ ]:
def assign_risk(p, low=LOW_THRESHOLD, high=HIGH_THRESHOLD):
    if p < low: return 'Low'
    if p < high: return 'Medium'
    return 'High'

test_proba_final = SELECTED_MODEL.predict_proba(X_test)[:, 1]
test_pred_final = (test_proba_final >= SELECTED_THRESHOLD).astype(int)  # operational threshold from validation
risk_labels = [assign_risk(p) for p in test_proba_final]
pred_df = pd.DataFrame({
    'invoice_id': test_df['invoice_id'].values,
    'customer_id': test_df['customer_id'].values,
    'invoice_date': test_df['invoice_date'].dt.strftime('%Y-%m-%d').values,
    'invoice_amount': test_df['invoice_amount'].values,
    'late_probability': np.round(test_proba_final, 4),
    'predicted_is_late': test_pred_final,
    'risk_category': risk_labels,
})
print(f'Predictions (test, {SELECTED_MODEL_NAME}, threshold={SELECTED_THRESHOLD:.2f}): {pred_df.shape}')
display(pred_df.head(10))
print(pred_df['risk_category'].value_counts())
print((pred_df['risk_category'].value_counts(normalize=True)*100).round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(test_proba_final, bins=30, edgecolor='black')
axes[0].axvline(LOW_THRESHOLD, color='green', linestyle='--', label=f'LOW {LOW_THRESHOLD}')
axes[0].axvline(HIGH_THRESHOLD, color='red', linestyle='--', label=f'HIGH {HIGH_THRESHOLD}')
axes[0].axvline(SELECTED_THRESHOLD, color='k', linestyle=':', label=f'decision {SELECTED_THRESHOLD:.2f}')
axes[0].set_title('Distribution of late_probability (test)'); axes[0].set_xlabel('P(is_late=1)'); axes[0].legend()
pred_df['risk_category'].value_counts().reindex(['Low','Medium','High']).plot(kind='bar', ax=axes[1], rot=0)
axes[1].set_title('Risk category counts (test)'); axes[1].set_ylabel('Invoices')
plt.tight_layout(); plt.show()
OPERATIONAL_TEST_METRICS = compute_binary_metrics(y_test, test_pred_final, test_proba_final)
print(f'Test metrics at operational threshold {SELECTED_THRESHOLD:.2f} (val-tuned, applied once):')
for k, v in OPERATIONAL_TEST_METRICS.items(): print(f"  {k:9s}: {v:.4f}")

## 16. Save Model

Save the selected pipeline (preprocessing + classifier) to `models/late_payment_classifier.joblib` and its metadata JSON.
No sensitive information is stored.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(SELECTED_MODEL, MODEL_PATH)
print('Saved model:', MODEL_PATH)
metadata = {
    'model_name': f'late_payment_{SELECTED_MODEL_NAME.lower().replace(" ", "_")}',
    'algorithm': SELECTED_MODEL_NAME,
    'feature_names': FEATURE_COLS,
    'training_rows': int(len(X_train)),
    'validation_rows': int(len(X_val)),
    'test_rows': int(len(X_test)),
    'training_date_range': [str(train_df['invoice_date'].min().date()), str(train_df['invoice_date'].max().date())],
    'validation_date_range': [str(validation_df['invoice_date'].min().date()), str(validation_df['invoice_date'].max().date())],
    'test_date_range': [str(test_df['invoice_date'].min().date()), str(test_df['invoice_date'].max().date())],
    'validation_metrics': {k: round(float(v), 4) for k, v in SELECTED_VAL.items()},
    'test_metrics': {k: round(float(v), 4) for k, v in TEST_METRICS_DEFAULT.items()},
    'test_metrics_operational_threshold': {k: round(float(v), 4) for k, v in OPERATIONAL_TEST_METRICS.items()},
    'classification_threshold': float(SELECTED_THRESHOLD),
    'risk_thresholds': {'LOW_THRESHOLD': float(LOW_THRESHOLD), 'HIGH_THRESHOLD': float(HIGH_THRESHOLD)},
    'dataset_version': DATASET_VERSION,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2))
print('Saved metadata:', METADATA_PATH)
print(json.dumps(metadata, indent=2))

## 17. Save Evaluation Results

Persist `reports/late_payment_model_comparison.csv` (`model,accuracy,precision,recall,f1,roc_auc,pr_auc`),
`reports/late_payment_test_metrics.json`, and the identification-safe prediction file.

In [ ]:
COMP_PATH.parent.mkdir(parents=True, exist_ok=True)
PRED_PATH.parent.mkdir(parents=True, exist_ok=True)
comparison_out = comparison_df.copy()
comparison_out.columns = ['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
comparison_out.to_csv(COMP_PATH, index=False)
print('Saved:', COMP_PATH); display(comparison_out.round(4))
test_metrics_out = {'model': SELECTED_MODEL_NAME, 'threshold_default_0_5': {k: round(float(v),4) for k,v in TEST_METRICS_DEFAULT.items()},
                    'threshold_operational': float(SELECTED_THRESHOLD),
                    'operational': {k: round(float(v),4) for k,v in OPERATIONAL_TEST_METRICS.items()}}
TEST_METRICS_PATH.write_text(json.dumps(test_metrics_out, indent=2))
print('Saved:', TEST_METRICS_PATH)
pred_df.to_csv(PRED_PATH, index=False)
print('Saved:', PRED_PATH, pred_df.shape)
print(pred_df.head(3).to_string(index=False))

## 18. Final Model Summary

In [ ]:
late_rate = 100 * float((train_df['is_late'] == 1).mean())
risk_counts = pred_df['risk_category'].value_counts()
def line(name):
    m = VAL_RESULTS[name]
    return f"{name}: Acc={m['accuracy']:.4f} Prec={m['precision']:.4f} Rec={m['recall']:.4f} F1={m['f1']:.4f} ROC={m['roc_auc']:.4f} PR={m['pr_auc']:.4f}"
print('==================================================')
print('FLOWCRED AI — LATE PAYMENT CLASSIFICATION')
print('==================================================')
print(f'Training invoices: {len(train_df)}')
print(f'Validation invoices: {len(validation_df)}')
print(f'Test invoices: {len(test_df)}')
print(f'Late-payment rate: {late_rate:.2f}%')
print('--------------------------------------------------')
print('MODEL COMPARISON')
print('--------------------------------------------------')
print('Dummy Baseline:', line('Dummy Baseline').replace('Dummy Baseline: ', '', 1))
print('Logistic Regression:', line('Logistic Regression').replace('Logistic Regression: ', '', 1))
print('Random Forest:', line('Random Forest').replace('Random Forest: ', '', 1))
print('XGBoost:', line('XGBoost').replace('XGBoost: ', '', 1))
print('--------------------------------------------------')
print('SELECTED MODEL')
print('--------------------------------------------------')
print(f'Model: {SELECTED_MODEL_NAME}')
print(f"Validation PR-AUC: {SELECTED_VAL['pr_auc']:.4f}")
print(f"Validation Recall: {SELECTED_VAL['recall']:.4f}")
print(f"Validation F1: {SELECTED_VAL['f1']:.4f}")
print(f"Validation ROC-AUC: {SELECTED_VAL['roc_auc']:.4f}")
print('--------------------------------------------------')
print('FINAL TEST PERFORMANCE')
print('--------------------------------------------------')
print(f"Test PR-AUC: {TEST_METRICS_DEFAULT['pr_auc']:.4f}")
print(f"Test Recall: {TEST_METRICS_DEFAULT['recall']:.4f}")
print(f"Test F1: {TEST_METRICS_DEFAULT['f1']:.4f}")
print(f"Test ROC-AUC: {TEST_METRICS_DEFAULT['roc_auc']:.4f}")
print('--------------------------------------------------')
print('THRESHOLD')
print('--------------------------------------------------')
print(f'Selected classification threshold: {SELECTED_THRESHOLD:.2f} (max validation F1; business validation still required)')
print('--------------------------------------------------')
print('RISK CATEGORIES')
print('--------------------------------------------------')
print(f"Low: {int(risk_counts.get('Low', 0))}")
print(f"Medium: {int(risk_counts.get('Medium', 0))}")
print(f"High: {int(risk_counts.get('High', 0))}")
print('--------------------------------------------------')
print('ARTIFACTS')
print('--------------------------------------------------')
print(f'Model: {MODEL_PATH}')
print(f'Metadata: {METADATA_PATH}')
print(f'Predictions: {PRED_PATH}')
print(f'Evaluation: {COMP_PATH}, {TEST_METRICS_PATH}')
print('==================================================')